# TP2 sur Colab : entraîner, évaluer et sauvegarder un modèle

Variante du TP2 pour les postes sans installation. Mêmes étapes, mêmes contrôles.

**Avant de commencer :** clonez votre dépôt du TP1, puis téléversez le kit TP2 (`tp2-semaine2.zip`)
avec l'icône dossier du panneau de gauche.


## Étape A. Récupérer le dépôt et installer le kit (contrôle C1)


In [ ]:
ORGANISATION = "larhlimi2023"   # votre compte ou organisation
DEPOT = "sentiment-app"

!git clone https://github.com/{ORGANISATION}/{DEPOT}.git
!unzip -q -o tp2-semaine2.zip
!cd tp2-semaine2 && python appliquer_tp2.py ../{DEPOT}
%cd {DEPOT}


In [ ]:
!pip install -q pytest pyarrow
!python -m pytest -q --deselect tests/test_environment.py::test_virtualenv_active
# attendu : 2 failed, 37 passed, 2 skipped (test_virtualenv_active écarté sur Colab)


## Étape B. Générer le jeu de démonstration (contrôle C2)


In [ ]:
!python src/make_demo.py
import pandas as pd
df = pd.read_csv('data/demo_reviews.csv')
print(df.label.value_counts())
df.sample(5, random_state=0)


Repérez un avis mitigé : une première moitié d'un sentiment, un retournement, une seconde moitié de l'autre.
L'étiquette suit la seconde moitié. Cette structure explique toutes les erreurs de l'étape D.


## Étape C. Entraîner et sauvegarder (contrôle C3)


In [ ]:
!python src/train.py
# attendu : F1 macro 0.879 sur le jeu de démonstration


In [ ]:
!cat models/metrics.json


## Étape D. Évaluer et lire les erreurs (contrôle C4)


In [ ]:
!python src/evaluate.py
# attendu : 7 erreurs sur 58 avis, toutes sur des avis mitigés


Écrivez trois phrases dans `docs/semaine2.md` : quelles erreurs, pourquoi, et ce qu'il faudrait pour les corriger.


## Étape E. Le vrai jeu, les tests, le push (contrôle C5)


In [ ]:
!python src/train.py --allocine
# 20 000 avis publics, attendu : F1 macro 0.920


In [ ]:
!python -m pytest -q --deselect tests/test_environment.py::test_virtualenv_active
# attendu : 41 passed (les 42 du TP moins test_virtualenv_active)


In [ ]:
from getpass import getpass
JETON = getpass("Jeton d'accès personnel GitHub : ")

!git config user.name "Prénom Nom" && git config user.email "prenom.nom@ecole.ma"
!git add -A && git commit -q -m "Semaine 2 : modele de sentiment, metriques et tests"
!git remote set-url origin https://{JETON}@github.com/{ORGANISATION}/{DEPOT}.git
!git push


## Les deux différences avec un poste local

**Pas d'environnement virtuel :** `test_virtualenv_active` est écarté, comme au TP1.

**Rien ne survit à la session :** poussez sur GitHub avant de fermer Colab, sinon le travail est perdu.
Le modèle `models/sentiment.joblib`, lui, n'a pas à être conservé : il se réentraîne en une commande.
